# Houston Model Tuning Workflow

This notebook performs simple sample-level model tuning using the current Houston CSV samples. It uses the existing repo config and package modules.

Important: the provided `sample == test` rows are held out for final evaluation. Tuning is done only from the provided training rows.

In [1]:
from pathlib import Path
import pandas as pd

sample_path = Path("/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/Houston_fl_samples.csv")
out_path = sample_path.with_name("Houston_fl_samples_train_test_swapped.csv")

samples = pd.read_csv(sample_path)

if "sample" not in samples.columns:
    raise ValueError("Column 'sample' not found.")

swap_map = {
    "train": "test",
    "test": "train",
}

samples["sample"] = (
    samples["sample"]
    .astype(str)
    .str.lower()
    .map(swap_map)
)

if samples["sample"].isna().any():
    bad_values = samples.loc[samples["sample"].isna(), "sample"].unique()
    raise ValueError(f"Unexpected sample values: {bad_values}")

samples.to_csv(out_path, index=False)

print("Saved:", out_path)
print(samples["sample"].value_counts())

Saved: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/Houston_fl_samples_train_test_swapped.csv
sample
train    3150
test      727
Name: count, dtype: int64


## 1. Setup

In [2]:
from pathlib import Path
import json
import sys

import numpy as np
import pandas as pd

REPO = Path.cwd()
if not (REPO / "src").exists():
    if (REPO.parent / "src").exists():
        REPO = REPO.parent
    else:
        REPO = Path("/home/abdullah/SAR-Urban-Flood-GEE-ML")

sys.path.insert(0, str(REPO / "src"))

from sar_flood_ml.config import load_config, output_dir, sample_columns
from sar_flood_ml.evaluate import evaluate_models
from sar_flood_ml.inference import predict_raster
from sar_flood_ml.models import label_maps, save_model_artifact, train_model
from sar_flood_ml.train import load_training_data

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

cfg = load_config(REPO / "configs/config.yaml")

# Use a fresh output folder so Windows file locks in outputs_local do not interrupt tuning.
cfg["paths"]["output_dir"] = "/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models"
OUT = output_dir(cfg)

print("Repo:", REPO)
print("Output:", OUT)

Repo: /home/abdullah/SAR-Urban-Flood-GEE-ML
Output: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models


## 2. Load Configured Training Data

In [3]:
X_train, X_test, y_train, y_test, samples, feature_names, classes = load_training_data(cfg)
cols = sample_columns(cfg)
flood_class = int(cols["flood_class"])

X_train_np = X_train.to_numpy(dtype="float32")
X_test_np = X_test.to_numpy(dtype="float32")
y_train_np = y_train.to_numpy(dtype="int32")
y_test_np = y_test.to_numpy(dtype="int32")

print("Features:", feature_names)
print("Classes:", classes)
print("Flood class:", flood_class)
print("Train:", X_train_np.shape)
print("Test:", X_test_np.shape)
display(pd.crosstab(samples[cols["split"]], samples[cols["label"]], margins=True))

Features: ['VV', 'VV_1', 'VH', 'VH_1']
Classes: [1, 2]
Flood class: 1
Train: (3150, 4)
Test: (727, 4)


classvalue,1,2,All
sample,,,
test,521,206,727
train,2612,538,3150
All,3133,744,3877


## 3. Tune Random Forest

Cross-validation happens inside the provided training rows only. The provided test rows are not used here.

In [4]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, make_scorer
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
flood_f1 = make_scorer(f1_score, pos_label=flood_class, zero_division=0)

rf_params = {
    "n_estimators": [5, 10, 20, 30, 40, 50, 60, 70, 80, 100, 120, 150, 200],
    "max_depth": [None, 5, 10, 20, 40],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4, 8],
    "max_features": ["sqrt", "log2", None],
    "class_weight": ["balanced", "balanced_subsample"],
}

# This grid has thousands of possible combinations. RandomizedSearchCV samples
# a controlled number of combinations so the notebook stays practical.
RF_N_ITER = 120

rf_search = RandomizedSearchCV(
    estimator=RandomForestClassifier(random_state=RANDOM_STATE, n_jobs=-1),
    param_distributions=rf_params,
    n_iter=RF_N_ITER,
    scoring=flood_f1,
    cv=cv,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
    return_train_score=True,
)

rf_search.fit(X_train_np, y_train_np)
rf_best = rf_search.best_estimator_

print("Best RF CV flood F1:", rf_search.best_score_)
print("Best RF params:", rf_search.best_params_)

rf_results = pd.DataFrame(rf_search.cv_results_).sort_values("rank_test_score")
rf_results.to_csv(OUT / "rf_tuning_cv_results.csv", index=False)
display(rf_results[["rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "params"]].head(10))

Fitting 5 folds for each of 120 candidates, totalling 600 fits
Best RF CV flood F1: 0.9350776179045704
Best RF params: {'n_estimators': 200, 'min_samples_split': 5, 'min_samples_leaf': 2, 'max_features': 'sqrt', 'max_depth': 40, 'class_weight': 'balanced'}


,rank_test_score,mean_test_score,std_test_score,mean_train_score,params
5,1,0.935078,0.004997,0.986768,"{'n_estimators': 200, 'min_samples_split': 5, ..."
50,2,0.935044,0.005508,0.996397,"{'n_estimators': 120, 'min_samples_split': 5, ..."
32,3,0.934551,0.004948,0.996782,"{'n_estimators': 120, 'min_samples_split': 5, ..."
117,3,0.934551,0.004948,0.996782,"{'n_estimators': 120, 'min_samples_split': 5, ..."
3,5,0.933852,0.006772,0.990294,"{'n_estimators': 70, 'min_samples_split': 2, '..."
28,6,0.933061,0.004774,0.996591,"{'n_estimators': 80, 'min_samples_split': 5, '..."
105,7,0.933008,0.006036,0.989233,"{'n_estimators': 40, 'min_samples_split': 2, '..."
17,8,0.932870,0.003627,0.997167,"{'n_estimators': 200, 'min_samples_split': 5, ..."
83,9,0.932248,0.005732,1.000000,"{'n_estimators': 100, 'min_samples_split': 2, ..."
114,10,0.932221,0.005252,0.989949,"{'n_estimators': 150, 'min_samples_split': 5, ..."


## 4. Tune XGBoost

XGBoost uses zero-based labels internally. The final evaluation converts predictions back to the original class labels.

In [5]:
try:
    import xgboost as xgb
    HAS_XGB = True
except ImportError:
    HAS_XGB = False
    print("XGBoost not installed. Skipping XGBoost tuning.")

xgb_best = None

if HAS_XGB:
    label_to_index, index_to_label = label_maps(classes)
    y_train_idx = np.array([label_to_index[int(v)] for v in y_train_np], dtype="int32")
    flood_idx = label_to_index[flood_class]
    xgb_flood_f1 = make_scorer(f1_score, pos_label=flood_idx, zero_division=0)

    xgb_params = {
        "n_estimators": [100, 200, 300, 500, 800],
        "max_depth": [2, 3, 4, 5, 6],
        "learning_rate": [0.01, 0.03, 0.05, 0.1],
        "subsample": [0.7, 0.85, 1.0],
        "colsample_bytree": [0.7, 0.85, 1.0],
        "min_child_weight": [1, 3, 5, 10],
        "reg_lambda": [0.5, 1.0, 2.0, 5.0, 10.0],
        "gamma": [0, 0.1, 0.5, 1.0],
    }

    xgb_search = RandomizedSearchCV(
        estimator=xgb.XGBClassifier(
            objective="binary:logistic" if len(classes) == 2 else "multi:softprob",
            eval_metric="logloss" if len(classes) == 2 else "mlogloss",
            random_state=RANDOM_STATE,
            n_jobs=-1,
        ),
        param_distributions=xgb_params,
        n_iter=40,
        scoring=xgb_flood_f1,
        cv=cv,
        n_jobs=-1,
        random_state=RANDOM_STATE,
        verbose=1,
        return_train_score=True,
    )

    xgb_search.fit(X_train_np, y_train_idx)
    xgb_best = xgb_search.best_estimator_

    print("Best XGB CV flood F1:", xgb_search.best_score_)
    print("Best XGB params:", xgb_search.best_params_)

    xgb_results = pd.DataFrame(xgb_search.cv_results_).sort_values("rank_test_score")
    xgb_results.to_csv(OUT / "xgb_tuning_cv_results.csv", index=False)
    display(xgb_results[["rank_test_score", "mean_test_score", "std_test_score", "mean_train_score", "params"]].head(10))

XGBoost not installed. Skipping XGBoost tuning.


## 5. Tune 1D CNN Optional

The CNN is a 1D feature CNN over the selected SAR features. It is not a spatial CNN. This cell runs only if TensorFlow is installed in the notebook kernel.

In [6]:
try:
    import tensorflow as tf
    HAS_TF = True
    print("TensorFlow:", tf.__version__)
except ImportError:
    HAS_TF = False
    print("TensorFlow not installed. Install with: pip install tensorflow-cpu")

from sklearn.model_selection import StratifiedShuffleSplit

cnn_best = None
cnn_tuning = None

if HAS_TF:
    splitter = StratifiedShuffleSplit(n_splits=1, test_size=0.20, random_state=RANDOM_STATE)
    inner_train_idx, inner_val_idx = next(splitter.split(X_train_np, y_train_np))

    X_inner_train = X_train_np[inner_train_idx]
    y_inner_train = y_train_np[inner_train_idx]
    X_inner_val = X_train_np[inner_val_idx]
    y_inner_val = y_train_np[inner_val_idx]

    cnn_trials = [
        {"filters_1": 16, "filters_2": 32, "dense_units": 32, "dropout": 0.20, "learning_rate": 0.001, "batch_size": 32},
        {"filters_1": 32, "filters_2": 64, "dense_units": 64, "dropout": 0.30, "learning_rate": 0.001, "batch_size": 64},
        {"filters_1": 64, "filters_2": 128, "dense_units": 64, "dropout": 0.40, "learning_rate": 0.0005, "batch_size": 64},
        {"filters_1": 32, "filters_2": 64, "dense_units": 128, "dropout": 0.20, "learning_rate": 0.0005, "batch_size": 32},
    ]

    cnn_rows = []
    cnn_models = {}

    for i, trial in enumerate(cnn_trials, start=1):
        name = f"cnn_trial_{i}"
        cnn_cfg = {
            "random_state": RANDOM_STATE,
            "epochs": 200,
            "patience": 20,
            "verbose": 0,
            **trial,
        }

        model_info = train_model(
            "cnn_1d",
            X_inner_train,
            y_inner_train,
            classes,
            cnn_cfg,
            X_val=X_inner_val,
            y_val=y_inner_val,
        )

        metrics_i, _ = evaluate_models(
            {name: model_info},
            X_inner_train,
            y_inner_train,
            X_inner_val,
            y_inner_val,
            classes,
            flood_class,
        )

        row = metrics_i.iloc[0].to_dict()
        row["trial"] = name
        row["params"] = json.dumps(trial)
        cnn_rows.append(row)
        cnn_models[name] = model_info

    cnn_tuning = pd.DataFrame(cnn_rows).sort_values(["flood_f1", "macro_f1"], ascending=False)
    cnn_tuning.to_csv(OUT / "cnn_tuning_results.csv", index=False)
    display(cnn_tuning)

    best_cnn_trial = cnn_tuning.iloc[0]["trial"]
    cnn_best = cnn_models[best_cnn_trial]
    print("Best CNN:", best_cnn_trial)

I0000 00:00:1780511499.070305    6396 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1780511499.076193    6396 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1780511499.640006    6396 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1780511501.980675    6396 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:0

TensorFlow: 2.21.0


E0000 00:00:1780511503.035041    6396 cuda_executor.cc:1737] INTERNAL: CUDA Runtime error: Failed call to cudaGetRuntimeVersion: Error loading CUDA libraries. GPU will not be used.: Error loading CUDA libraries. GPU will not be used.
W0000 00:00:1780511503.035526   64852 cuda_executor.cc:1755] Failed to determine cuDNN version (Note that this is expected if the application doesn't link the cuDNN plugin): INTERNAL: cuDNN error: CUDNN_STATUS_INTERNAL_ERROR
W0000 00:00:1780511503.060663    6396 gpu_device.cc:2365] Cannot dlopen some GPU libraries. Please make sure the missing libraries mentioned above are installed properly if you would like to use GPU. Follow the guide at https://www.tensorflow.org/install/gpu for how to download and setup the required libraries for your platform.
Skipping registering GPU devices...


,model,accuracy,macro_f1,flood_precision,flood_recall,flood_f1,train_macro_f1,train_test_macro_f1_gap,trial,params
0,cnn_trial_1,0.852381,0.793210,0.982022,0.837165,0.903826,0.801373,0.008163,cnn_trial_1,"{""filters_1"": 16, ""filters_2"": 32, ""dense_unit..."
3,cnn_trial_4,0.846032,0.787324,0.984055,0.827586,0.899063,0.801817,0.014492,cnn_trial_4,"{""filters_1"": 32, ""filters_2"": 64, ""dense_unit..."
2,cnn_trial_3,0.841270,0.781250,0.981735,0.823755,0.895833,0.812287,0.031037,cnn_trial_3,"{""filters_1"": 64, ""filters_2"": 128, ""dense_uni..."
1,cnn_trial_2,0.836508,0.777221,0.983834,0.816092,0.892147,0.800038,0.022817,cnn_trial_2,"{""filters_1"": 32, ""filters_2"": 64, ""dense_unit..."


Best CNN: cnn_trial_1


## 6. Final Holdout Evaluation

This is the first time the tuned models are evaluated on the provided test rows.

In [7]:
tuned_models = {
    "random_forest_tuned": {
        "kind": "sklearn_labels",
        "model": rf_best,
    }
}

if xgb_best is not None:
    tuned_models["xgboost_tuned"] = {
        "kind": "indexed_labels",
        "model": xgb_best,
    }

if cnn_best is not None:
    tuned_models["cnn_1d_tuned"] = cnn_best

metrics, reports = evaluate_models(
    tuned_models,
    X_train_np,
    y_train_np,
    X_test_np,
    y_test_np,
    classes,
    flood_class,
)

display(metrics)

metrics_path = OUT / "tuned_model_metrics.csv"
reports_path = OUT / "tuned_classification_reports.json"
metrics.to_csv(metrics_path, index=False)
with reports_path.open("w", encoding="utf-8") as f:
    json.dump(reports, f, indent=2)

print("Saved metrics:", metrics_path)
print("Saved reports:", reports_path)

,model,accuracy,macro_f1,flood_precision,flood_recall,flood_f1,train_macro_f1,train_test_macro_f1_gap
1,cnn_1d_tuned,0.947730,0.936742,0.974460,0.952015,0.963107,0.799721,-0.137021
0,random_forest_tuned,0.942228,0.925686,0.936248,0.986564,0.960748,0.964146,0.038460


Saved metrics: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/tuned_model_metrics.csv
Saved reports: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/tuned_classification_reports.json


## 7. Save Tuned Model Artifacts

In [8]:
model_metadata = {
    "features": feature_names,
    "classes": classes,
    "label_column": cols["label"],
    "flood_class": flood_class,
    "note": "Models tuned using provided training rows only; final metrics use provided test rows.",
}

artifacts = {}
for model_name, model_info in tuned_models.items():
    artifacts[model_name] = save_model_artifact(model_name, model_info, OUT, model_metadata)

artifacts_path = OUT / "tuned_model_artifacts.json"
with artifacts_path.open("w", encoding="utf-8") as f:
    json.dump(artifacts, f, indent=2)

artifacts

{'random_forest_tuned': {'kind': 'sklearn_labels',
  'model_file': '/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/random_forest_tuned_model.joblib'},
 'cnn_1d_tuned': {'kind': 'cnn_1d',
  'model_file': '/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/cnn_1d_tuned_model.keras',
  'scaler_file': '/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/cnn_1d_tuned_scaler.joblib',
  'history_csv': '/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/cnn_1d_tuned_training_history.csv'}}

## 8. Create Flood Maps From Tuned Models

In [9]:
prediction_outputs = predict_raster(
    cfg,
    tuned_models,
    feature_names,
    classes,
    flood_class,
    OUT,
)

prediction_outputs_path = OUT / "tuned_prediction_outputs.json"
with prediction_outputs_path.open("w", encoding="utf-8") as f:
    json.dump(prediction_outputs, f, indent=2)

prediction_outputs

{'random_forest_tuned': {'class_map': '/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/random_forest_tuned_class_map.tif',
  'flood_probability_map': '/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/random_forest_tuned_flood_probability.tif',
  'valid_pixels': 188266},
 'cnn_1d_tuned': {'class_map': '/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/cnn_1d_tuned_class_map.tif',
  'flood_probability_map': '/mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models/cnn_1d_tuned_flood_probability.tif',
  'valid_pixels': 188266}}

## 9. Output Files

In [10]:
print("Output folder:", OUT)
for path in sorted(OUT.glob("*")):
    print("-", path.name)

Output folder: /mnt/c/Users/amehedi/Downloads/SAR-Urban-Flood-GEE-M/outputs_tuned_models
- cnn_1d_tuned_class_map.tif
- cnn_1d_tuned_flood_probability.tif
- cnn_1d_tuned_model.keras
- cnn_1d_tuned_scaler.joblib
- cnn_1d_tuned_training_history.csv
- cnn_tuning_results.csv
- random_forest_tuned_class_map.tif
- random_forest_tuned_flood_probability.tif
- random_forest_tuned_model.joblib
- rf_focused_grid_cv_results.csv
- rf_tuning_cv_results.csv
- tuned_classification_reports.json
- tuned_model_artifacts.json
- tuned_model_metrics.csv
- tuned_prediction_outputs.json
